## House Prices — regression loyihasi

Bu safar Kaggle'ning House Prices dataseti bilan ishlayman. Maqsad — uy narxini (`SalePrice`) boshqa xususiyatlar asosida bashorat qilish. Avval datani Drive'dan yuklab olaman.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

path = '/content/drive/MyDrive/ML_Projects/Regression/house_prices'

train = pd.read_csv(f'{path}/train.csv')
test = pd.read_csv(f'{path}/test.csv')

train.head()

Mounted at /content/drive


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Endi dataga umumiy nazar tashlaymiz — nechta ustun bor, qaysi tipda.

In [2]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

Qaysi ustunlarda ma'lumot yetishmayotganini ko'raman. Faqat bo'sh joyi bor ustunlarni ko'rsataman, ko'p bo'lganidan kamiga qarab.

In [3]:
missing1 = train.isnull().sum()
missing1 = missing1[missing1>0]
missing1 = missing1.sort_values(ascending=False)
missing1

,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageType,81
GarageYrBlt,81
GarageFinish,81


`SalePrice` — bashorat qilmoqchi bo'lgan narsam. Statistikasiga qarayman.

In [4]:
train['SalePrice'].describe()

,SalePrice
count,1460.000000
mean,180921.195890
std,79442.502883
min,34900.000000
25%,129975.000000
50%,163000.000000
75%,214000.000000
max,755000.000000


Bu yerda bitta muhim narsa bor: ko'p ustunda "NA" aslida bo'sh joy emas, balki "bu narsa uyda yo'q" degani (masalan garaj yoki bassein yo'q). Shuning uchun bu ustunlarni tozalash o'rniga "None" deb to'ldiraman.

In [5]:
lab = ['PoolQC','Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2','MiscFeature','MasVnrType']
for i in range(len(lab)):
  train[lab[i]] = train[lab[i]].fillna('None')
train['GarageYrBlt'] = train['GarageYrBlt'].fillna(0)

In [6]:
train.isnull().sum().sort_values(ascending=False).head(10)

,0
LotFrontage,259
MasVnrArea,8
Electrical,1
Id,0
MSSubClass,0
Street,0
Alley,0
MSZoning,0
LotArea,0
Utilities,0


Qolgan ustunlar esa haqiqatan ham yetishmayotgan ma'lumot — bularni median/mode bilan to'ldiraman.

In [7]:
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])

In [8]:
train.isnull().sum().sort_values(ascending=False).head(5)

,0
Id,0
MSSubClass,0
MSZoning,0
LotFrontage,0
LotArea,0


In [9]:
train['LotFrontage'].dtype

dtype('float64')

Categorical ustunlarni raqamga aylantirish kerak, chunki model matn bilan ishlay olmaydi. `get_dummies` buni avtomatik qiladi.

In [10]:
train_encoded = pd.get_dummies(train,drop_first=True)
train_encoded.shape

(1460, 261)

`X` (feature'lar) va `y` (`SalePrice`)ni ajrataman. `Id` ustuni kerak emas, u shunchaki tartib raqami.

In [11]:
X = train_encoded.drop(['SalePrice','Id'],axis=1)
y = train_encoded['SalePrice']

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

### Birinchi model — Linear Regression

Eng oddiy modeldan boshlayman, barcha ustunlar bilan.

In [13]:
from sklearn.linear_model import LinearRegression
linearmodel = LinearRegression()
linearmodel.fit(X_train,y_train)
y_pred = linearmodel.predict(X_val)

In [14]:
from sklearn.metrics import r2_score,mean_squared_error
r2 = r2_score(y_val,y_pred)
mean = mean_squared_error(y_val,y_pred)
print(r2,mean)

0.0991450491073046 6909851338.706512


Natija juda past chiqdi (R² ≈ 0.1). Sabab — feature juda ko'p (260 atrofida), model chalkashib qolgan. Qaysi ustunlar narxga eng ko'p bog'liq ekanini ko'ray.

In [15]:
train.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False).head(15)

,SalePrice
SalePrice,1.000000
OverallQual,0.790982
GrLivArea,0.708624
GarageCars,0.640409
GarageArea,0.623431
TotalBsmtSF,0.613581
1stFlrSF,0.605852
FullBath,0.560664
TotRmsAbvGrd,0.533723
YearBuilt,0.522897


Faqat eng bog'liq ~14 ta ustun bilan qayta sinab ko'raman.

In [16]:
listx2 = ['OverallQual','GrLivArea','GarageCars','GarageArea','TotalBsmtSF','1stFlrSF','FullBath',
        'TotRmsAbvGrd','YearBuilt','YearRemodAdd','MasVnrArea','Fireplaces','BsmtFinSF1','LotFrontage']
X2 = train[listx2]
X2_train,X2_val,y_train,y_val = train_test_split(X2,y,test_size=0.2,random_state=42)

In [17]:
linearmodel2 = LinearRegression()
linearmodel2.fit(X2_train,y_train)
y2_pred = linearmodel2.predict(X2_val)
r22 = r2_score(y_val,y2_pred)
mse2 = mean_squared_error(y_val,y2_pred)
print(r22,mse2)

0.8113243870612996 1447203498.580808


### Ridge regression

Barcha ustunlarni saqlab, lekin ularning ta'sirini kamaytirib qo'yadigan usul. Shovqinli feature'lar kamroq ta'sir qiladi.

In [44]:
from sklearn.linear_model import Ridge
ridgemodel = Ridge(alpha=10)
ridgemodel.fit(X_train,y_train)
y_pred_ridge = ridgemodel.predict(X_val)
r2_ridge = r2_score(y_val,y_pred_ridge)
mse_ridge = mean_squared_error(y_val,y_pred_ridge)
print(r2_ridge,mse_ridge)

0.8780416594735395 935460255.5251619


In [47]:
from sklearn.linear_model import Ridge
ridgemodel = Ridge(alpha=5)
ridgemodel.fit(X_train,y_train)
y_pred_ridge = ridgemodel.predict(X_val)
r2_ridge = r2_score(y_val,y_pred_ridge)
mse_ridge = mean_squared_error(y_val,y_pred_ridge)
print(r2_ridge,mse_ridge)

0.8795871366840857 923605941.1744989


### Lasso regression

Ridge'ga o'xshaydi, lekin ba'zi foydasiz ustunlarni butunlay 0'ga tushirib yuboradi.

In [19]:
from sklearn.linear_model import Lasso
lassomodel = Lasso(alpha=10)
lassomodel.fit(X_train,y_train)
y_pred_lasso = lassomodel.predict(X_val)
r2_lasso = r2_score(y_val,y_pred_lasso)
mse_lasso = mean_squared_error(y_val,y_pred_lasso)
print(r2_lasso,mse_lasso)

0.839734296595269 1229290224.9385867


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.485e+11, tolerance: 6.967e+08
  model = cd_fast.enet_coordinate_descent(


Feature'lar turli shkalada (masalan yil vs maydon), shuning uchun scale qilib ko'raman — ba'zi modellar buni yaxshi ko'radi.

In [20]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [33]:
lassomodel_scaled = Lasso(alpha=1000)
lassomodel_scaled.fit(X_train_scaled,y_train)
y_pred_lasso_scaled = lassomodel_scaled.predict(X_val_scaled)
r2_lasso_scaled = r2_score(y_val,y_pred_lasso_scaled)
mse_lasso_scaled = mean_squared_error(y_val,y_pred_lasso_scaled)
print(r2_lasso_scaled,mse_lasso_scaled)

0.853514484149265 1123591691.2032175


In [40]:
ridgemodel_scaled = Ridge(alpha=10)
ridgemodel_scaled.fit(X_train_scaled,y_train)
y_pred_ridge_scaled = ridgemodel_scaled.predict(X_val_scaled)
r2_ridge_scaled = r2_score(y_val,y_pred_ridge_scaled)
mse_ridge_scaled = mean_squared_error(y_val,y_pred_ridge_scaled)
print(r2_ridge_scaled,mse_ridge_scaled)

0.8459399198580344 1181691070.1942673


### Eng yaxshi alpha'ni avtomatik topish

Qo'lda alpha sinash o'rniga, GridSearchCV bilan bir nechta qiymatni birdan sinab, eng yaxshisini topaman.

In [48]:
from sklearn.model_selection import GridSearchCV
param_grid = {'alpha': [0.1, 1, 3, 5, 7, 10, 15, 20]}
grid_search = GridSearchCV(Ridge(),param_grid,cv=5,scoring='r2')
grid_search = grid_search.fit(X_train,y_train)
print(grid_search.best_params_)

{'alpha': 10}


In [49]:
best_ridge = grid_search.best_estimator_
y_pred_best = best_ridge.predict(X_val)
r2_score(y_val, y_pred_best)

0.8780416594735395

## Xulosa

Turli modellarni sinab ko'rdim, natijalar quyidagicha:

| Model | R² |
|---|---|
| Linear Regression (barcha 259 ustun) | 0.099 |
| Linear Regression (14 ta eng muhim ustun) | 0.811 |
| Lasso (scale qilinmagan, alpha=10) | 0.840 |
| Ridge (scale qilingan, alpha=10) | 0.846 |
| Lasso (scale qilingan, alpha=1000) | 0.854 |
| Ridge (scale qilinmagan, alpha=10) | 0.878 |
| **Ridge (scale qilinmagan, alpha=5)** | **0.880** |

Eng yaxshi natijani Ridge berdi, alpha atrofida 5-10 bo'lganda. Eng katta dars — feature ko'p bo'lganda oddiy Linear Regression yaxshi ishlamaydi, regularization (Ridge/Lasso) yordam beradi.

In [ ]:
import matplotlib.pyplot as plt

models = ['Linear\n(259 ustun)', 'Linear\n(14 ustun)', 'Lasso\n(alpha=10)', 'Ridge\n(scaled)', 'Lasso\n(scaled)', 'Ridge\n(alpha=10)', 'Ridge\n(alpha=5)']
scores = [0.099, 0.811, 0.840, 0.846, 0.854, 0.878, 0.880]

plt.figure(figsize=(9, 5))
plt.bar(models, scores, color='#4C72B0')
plt.ylabel('R² natija')
plt.title('Modellarning solishtirilishi (R²)')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()